In [ ]:
import numpy as np
import pandas as pd
import cvxpy as cp
from pricing_lib.optimizer import PortfolioOptimizer
from scipy.stats import norm 
import statsmodels.api as sm
from scipy.stats import gaussian_kde, linregress, norm
import plotly.graph_objs as go
import plotly.figure_factory as ff 
import matplotlib.pyplot as plt
from numpy.linalg import inv as np_inv
from plotly.subplots import make_subplots
import seaborn as sns
from datetime import datetime, date

In [ ]:
# sns plot style 
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)
print(cp.installed_solvers())

In [ ]:
# Read Data
vix_options_df = pd.read_csv("./Data/vix_options_data.csv") 
vix_prices_df = pd.read_csv("./Data/vix_prices.csv") 

# Column Formatting 
vix_prices_df.rename(columns = {"DATE" : "date", "OPEN" : "px"}, inplace = True)
vix_options_df["sym"] = vix_options_df.underlying_symbol.apply(lambda x: x[1:]).values
vix_options_df.rename(columns = {"bid_size_1545": "bsize", 
                           "ask_size_1545" : "asize", 
                           "bid_1545": "bid", 
                           "ask_1545":"ask"}, inplace = True)
vix_options_df = vix_options_df[["quote_date", "expiration", "sym", "option_type", "strike", "bsize", "bid", "ask", "asize"]]

In [ ]:
# Process the VIX prices data
vix_prices_df['date'] = pd.to_datetime(vix_prices_df['date'])
vix_prices_df = vix_prices_df.sort_values(by='date')
vix_prices_df['VIX_log'] = np.log(vix_prices_df['px'])
vix_prices_df['delta_xi'] = vix_prices_df['VIX_log'].diff()
vix_prices_df['xi_lagged'] = vix_prices_df['VIX_log'].shift(1)
vix_prices_df = vix_prices_df.dropna()

# Performing OLS
regression = linregress(vix_prices_df['xi_lagged'], vix_prices_df['delta_xi'])
beta_1 = regression.slope
beta_0 = regression.intercept
pred_values  = beta_1*vix_prices_df.xi_lagged.values + beta_0
obs_values = vix_prices_df.delta_xi.values
residuals = obs_values - pred_values

# Calculate parameters a, xi_bar, and sigma
a = -beta_1
xi_bar = beta_0 / a
sigma = np.std(residuals)

print(f"OU Model Parameters:")
print("-" * 40)
print(f"{'a':<10} | {'xi_bar':<10} | {'sigma':<10}")
print("-" * 40)
print(f"{a:<10.5f} | {xi_bar:<10.5f} | {sigma:<10.5f}")
print("-" * 40)

# Calculate date to maturity
start_date = date(2024, 6, 21) # t = 0
end_date = date(2024, 7, 17) # t = T
days = pd.date_range(start = start_date, end = end_date, freq = "B")
T = len(days)-1

#VIX value at initial date
VIX_0 = 13.20
xi_0 = np.log(VIX_0)

# Calculate the mean, median and std. dev. of xi_T
phi = 1 - a
mu_xi_T = xi_bar + (xi_0 - xi_bar)*(phi**T)
std_xi_T = np.sqrt(sigma**2 * (1 - phi**(2*T)) / (1 - phi**2))
median_vix_T = np.exp(mu_xi_T)

print(f"Time to maturity (T) is {T} days.")
print(f"Mean and standard deviation of log-VIX at maturity:")
print("-" * 40)
print(f"{'E[xi_T]':<15} | {'sigma(xi_T)':<15}")
print("-" * 40)
print(f"{mu_xi_T:<15.4f} | {std_xi_T:<15.6f}")
print("-" * 40)
print(f"Median of VIX at maturity: {median_vix_T:.6f}")

In [ ]:
# Task 4: solving optimization for lambda = 2 and 20
NO_SCENARIOS = 100
portfolio_1 =  PortfolioOptimizer(lob_data=vix_options_df.copy(), 
                                price_data=vix_prices_df.copy(),
                                wealth = 1e5,
                                w_hat = 1e5,
                                init_claim = 0.0,
                                claim_sign = 1,
                                claim_strike = None,
                                risk_param = 2, 
                                mu = mu_xi_T, 
                                std = std_xi_T, 
                                no_scenarios = NO_SCENARIOS)
portfolio_1.initialize_params()

portfolio_2 =  PortfolioOptimizer(lob_data=vix_options_df, 
                                price_data=vix_prices_df,
                                wealth = 1e5,
                                w_hat = 1e5,
                                init_claim = 0.0,
                                claim_sign = 1,
                                claim_strike = None,
                                risk_param = 20, 
                                mu = mu_xi_T, 
                                std = std_xi_T, 
                                no_scenarios = NO_SCENARIOS)

portfolio_2.initialize_params()
portfolio_1.solve()
portfolio_2.solve()

In [ ]:
# Plot terminal wealth vs terminal VIX scenarios
vix_px= portfolio_1.get_vix_prices()
terminal_wealth_2 = portfolio_1.get_terminal_wealth()
terminal_wealth_20 = portfolio_2.get_terminal_wealth()

fig = make_subplots(rows = 1, cols = 1, subplot_titles = ['Terminal Wealth vs VIX(T)'])
fig.add_trace(go.Scatter(y = terminal_wealth_2, x = vix_px, name = "lambda = 2"))
fig.add_trace(go.Scatter(y = terminal_wealth_20, x = vix_px, name = "lambda = 20"))
fig.add_vline(x = median_vix_T, line_dash = "dash", 
              annotation_text="Median VIX(T) ", 
              annotation_position="bottom right")
fig.update_yaxes(title_text="<b> Terminal Wealth  </b>", dtick = 2e5, tickprefix = '$', row=1, col=1,)
fig.update_xaxes(title_text="<b> VIX Price at Maturity  </b>", row=1, col=1 )
fig.update_layout(height = 600)
fig.show()

In [ ]:
# KDE
kde_plot, ax = plt.subplots()
pal = sns.color_palette()
sns.kdeplot(terminal_wealth_2,  ax=ax, color=pal[0], label='lambda = 2')
sns.kdeplot(terminal_wealth_20, ax=ax, color=pal[1], label='lambda = 20')

# Histogram
ax.hist(terminal_wealth_2, density=True, alpha = 0.4)
ax.hist(terminal_wealth_20, density=True, alpha = 0.4)
ax.set_xlabel("Terminal Wealth at Maturity")
ax.set_ylabel("Kernel Density Estimate")
ax.legend()
plt.show()

In [ ]:
# Summary Statistics
pd.options.display.float_format = '{:,.2f}'.format
summary_df = pd.DataFrame({"terminal_wealth_2": terminal_wealth_2, "terminal_wealth_20": terminal_wealth_20 })
summary_df = summary_df.agg(["min", "max", "mean","median", "std" ,"kurtosis", "skew"]).transpose()
cash_pos_2 = portfolio_1.w - portfolio_1.Sx.value
cash_pos_20 = portfolio_2.w - portfolio_2.Sx.value
summary_df["Cash_Allocation"] = [cash_pos_2, cash_pos_20]
summary_df

In [ ]:
def get_indifference_price( init_claim, strike, w_hat, side):
    """
    Function to evaluate indifferet price 
    """
    NO_SCENARIOS = 100
    RISK_PARAM = 20
    if side == "BUY": # BUY CLAIM
        # Base portfolio
        P1 =  PortfolioOptimizer(lob_data=vix_options_df.copy(),price_data=vix_prices_df.copy(),
                                        wealth = 0.0, w_hat = w_hat,
                                        init_claim = init_claim, claim_sign = 1, claim_strike = None,
                                        risk_param = RISK_PARAM, mu = mu_xi_T, std = std_xi_T, 
                                        no_scenarios = NO_SCENARIOS)
        # Portfolio that purchases claim
        P2 =  PortfolioOptimizer(lob_data=vix_options_df.copy(),price_data=vix_prices_df.copy(),
                                        wealth = 0.0, w_hat = w_hat,
                                        init_claim = init_claim, claim_sign = -1, claim_strike = strike,
                                        risk_param = RISK_PARAM, mu = mu_xi_T, std = std_xi_T, 
                                        no_scenarios = NO_SCENARIOS)

    elif side == "SELL":  # SELL CLAIM
        # portfolio that sells claim
        P1 =  PortfolioOptimizer(lob_data=vix_options_df.copy(),price_data=vix_prices_df.copy(),
                                        wealth = 0.0, w_hat = w_hat,
                                        init_claim = init_claim, claim_sign = 1, claim_strike = strike,
                                        risk_param = RISK_PARAM, mu = mu_xi_T, std = std_xi_T,  
                                        no_scenarios = NO_SCENARIOS)
        # Base portfolio
        P2 =  PortfolioOptimizer(lob_data=vix_options_df.copy(),price_data=vix_prices_df.copy(),
                                        wealth = 0.0, w_hat = w_hat,
                                        init_claim = init_claim, claim_sign = 1, claim_strike = None,
                                        risk_param = RISK_PARAM,  mu = mu_xi_T, std = std_xi_T, 
                                        no_scenarios = NO_SCENARIOS)
    else:
        raise ValueError("Side given isn't valid")
    
    P1.initialize_params()
    P2.initialize_params()

    P1.solve()
    P2.solve()
    
    indifference_px = w_hat*(P2.score() - P1.score())
    return indifference_px


In [ ]:
# Compute Indifference BUY/SELL prices for option strikes
strikes = [5*i for i in range(11)]
buy_prices = []
sell_prices = []
for K in strikes:
    buy_px = get_indifference_price(init_claim=0, strike=K, w_hat=1e5, side = "BUY")
    buy_prices.append(-1*buy_px)
    sell_px = get_indifference_price(init_claim=0, strike=K, w_hat=1e5, side = "SELL")
    sell_prices.append(-1*sell_px)
  

In [ ]:
# Plot buy-sell prices vs option strike 
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=strikes, y=buy_prices, mode='lines', name='BUY'))
fig2.add_trace(go.Scatter(x=strikes, y=sell_prices, mode='lines', name='SELL'))
fig2.update_layout(
    title="<b>Indifference price on Digital Option</b>",
    title_x = 0.5,
    xaxis_title="Digital Option Strike",
    yaxis_title="Indifference Price",
)
fig2.show()

In [ ]:
# Plot indifference price spread
spread = np.subtract(sell_prices, buy_prices)
fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=strikes, y=spread, mode='lines', name='BUY'))
fig3.add_vline(x = median_vix_T, line_dash = "dash", 
              annotation_text="Med VIX(T) ", 
              annotation_position="bottom right")
fig3.update_layout(
    title="<b>Indifference Price Buy-Sell Spread </b>",
    title_x = 0.5, 
    xaxis_title="<b> Option Strike Price</b>",
    yaxis_title="<b> Buy-Sell Spread </b>",
)
fig3.show()

In [ ]:
P = portfolio_1.P
px = portfolio_1.get_vix_prices()
positions = portfolio_1.get_positions()
weights = portfolio_1.get_terminal_weights()
log_prices = portfolio_1.get_terminal_price_grid()
prices =  portfolio_1.get_vix_prices()

In [ ]:
fig4 = go.Figure()
fig4.add_trace(go.Scatter(x=log_prices, y=weights, mode='lines', name='BUY'))
fig4.update_layout(
    title="Terminal Price Distribution",
    xaxis_title="log VIX(T) prices",
    yaxis_title="P.D.F weight",
)
fig4.show()

In [ ]:
fig5 = go.Figure()
fig5.add_trace(go.Scatter(x=prices, y=weights, mode='lines', name='BUY'))
fig5.update_layout(
    title="Terminal Price Distribution",
    xaxis_title="log VIX(T) prices",
    yaxis_title="P.D.F weight", 
)
fig5.show()

# Part 3: Quadratic Rough Heston

Replaces the discrete OU model for log-VIX with the calibrated QRH model of Bourgey & Gatheral (2026).
Scenarios now come from Monte Carlo simulation of SPX and VIX jointly rather than a 1d quadrature, and
the tradable set is extended from VIX options alone to SPX **and** VIX quotes at the 17/07/2024 expiry.

In [ ]:
from pricing_lib.qrh_calibration import QRH_Calibration
from pricing_lib.qrh_optimizer import PortfolioOptimizerQRH, process_snapshot_data
from pricing_lib.qrh_utils import RunParams, init_logging

init_logging()
SNAPSHOT = "./Data/spx_vix_snapshot.csv"

# Calibrate QRH to the same 21/06/2024 snapshot (SPX 17/07 + first four liquid VIX expiries)
run_params = RunParams(data_path=SNAPSHOT, quote_dt="2024-06-21", expiry_dt="2024-07-17",
                       n_paths=60_000, workers=4, tag="qrh")
qrh = QRH_Calibration(run_params=run_params)
qrh.run()

In [ ]:
# Tradables for the 17/07/2024 expiry
vix_only_df = process_snapshot_data(SNAPSHOT, expiry="2024-07-17", symbols=("VIX",))
spx_vix_df  = process_snapshot_data(SNAPSHOT, expiry="2024-07-17")

# the VIX leg must be the identical contract set to Part 2 for the comparison to mean anything
same = set(zip(vix_options_df.option_type, vix_options_df.strike)) == \
       set(zip(vix_only_df.option_type,    vix_only_df.strike))
print(f"Part 2 VIX tradables: {len(vix_options_df)}   Part 3 VIX tradables: {len(vix_only_df)}"
      f"   identical set: {same}")
print(spx_vix_df.groupby("sym").agg(n=("strike", "size"), lo=("strike", "min"), hi=("strike", "max")))
spx_vix_df.head()

In [ ]:
# Task 4 (QRH): solving optimization for lambda = 2 and 20
NO_SCENARIOS_QRH = 5000
portfolio_1_qrh = PortfolioOptimizerQRH(lob_data=spx_vix_df.copy(),
                                qrh_calibration = qrh,
                                wealth = 1e5,
                                w_hat = 1e5,
                                init_claim = 0.0,
                                claim_sign = 1,
                                claim_strike = None,
                                risk_param = 2,
                                no_scenarios = NO_SCENARIOS_QRH)
portfolio_1_qrh.initialize_params()

portfolio_2_qrh = PortfolioOptimizerQRH(lob_data=spx_vix_df.copy(),
                                qrh_calibration = qrh,
                                wealth = 1e5,
                                w_hat = 1e5,
                                init_claim = 0.0,
                                claim_sign = 1,
                                claim_strike = None,
                                risk_param = 20,
                                no_scenarios = NO_SCENARIOS_QRH)
portfolio_2_qrh.initialize_params()
portfolio_1_qrh.solve()
portfolio_2_qrh.solve()

print(f"lambda=2  status={portfolio_1_qrh.PROBLEM.status}  score={portfolio_1_qrh.score():.5f}")
print(f"lambda=20 status={portfolio_2_qrh.PROBLEM.status}  score={portfolio_2_qrh.score():.5f}")

In [ ]:
# Optimal positions under QRH, with the quoted depth they trade against
for name, p in [("lambda = 2", portfolio_1_qrh), ("lambda = 20", portfolio_2_qrh)]:
    pos = pd.DataFrame({"sym": spx_vix_df.sym, "option_type": spx_vix_df.option_type,
                        "strike": spx_vix_df.strike,
                        "bid": spx_vix_df.bid, "bsize": spx_vix_df.bsize,
                        "ask": spx_vix_df.ask, "asize": spx_vix_df.asize,
                        "position": p.get_positions()})
    # x_plus is capped by asize (buying), x_minus by bsize (selling), so the
    # relevant depth depends on the sign of the position
    pos["depth"] = np.where(pos.position >= 0, pos.asize, pos.bsize)
    pos["pct_of_depth"] = 100 * np.abs(pos.position) / pos.depth

    active = pos[np.abs(pos.position) > 1e-6].copy()
    at_limit = (active.pct_of_depth > 99.9).sum()
    print(f"{name}: {len(active)} active of {p.no_assets}   cost Sx = {p.Sx.value:,.0f}"
          f"   at the size limit: {at_limit}")
    print(active.round(2).to_string(index=False), "\n")

In [ ]:
# Terminal wealth is now a function of BOTH underlyings, so plotting it against
# VIX alone collapses the SPX dimension into vertical scatter. Surface over both.
from scipy.interpolate import griddata

fig4 = make_subplots(rows=1, cols=2,
                     specs=[[{"type": "surface"}, {"type": "surface"}]],
                     subplot_titles=("lambda = 2", "lambda = 20"))

for col, p in enumerate([portfolio_1_qrh, portfolio_2_qrh], start=1):
    vix, spx, wealth = p.get_vix_prices(), p.get_spx_prices(), p.get_terminal_wealth()
    # interpolate the scattered paths onto a regular grid over the bulk of the
    print(f" Portfolio {col} (lambda ={p._lambda})| max wealth: {max(wealth):,} | min wealth: {min(wealth):,} ")
    vi = np.linspace(np.percentile(vix, 1), np.percentile(vix, 99), 60)
    si = np.linspace(np.percentile(spx, 1), np.percentile(spx, 99), 60)
    VI, SI = np.meshgrid(vi, si)
    Z = griddata((vix, spx), wealth, (VI, SI), method="linear")
    fig4.add_trace(go.Surface(x=VI, y=SI, z=Z, colorscale="Viridis",
                              showscale=(col == 2), colorbar=dict(title="Wealth")),
                   row=1, col=col)

scene = dict(xaxis_title="VIX at maturity", yaxis_title="SPX at maturity",
             zaxis_title="Terminal wealth")
fig4.update_layout(title="<b>QRH terminal wealth vs (VIX, SPX) at maturity</b>",
                   title_x=0.5, height=650, scene=scene, scene2=scene)
fig4.show()

In [ ]:
# Same data as raw paths, coloured by wealth - shows where the paths actually are
fig4b = make_subplots(rows=1, cols=2,
                      specs=[[{"type": "scene"}, {"type": "scene"}]],
                      subplot_titles=("lambda = 2", "lambda = 20"))
for col, p in enumerate([portfolio_1_qrh, portfolio_2_qrh], start=1):
    vix, spx, wealth = p.get_vix_prices(), p.get_spx_prices(), p.get_terminal_wealth()
    fig4b.add_trace(go.Scatter3d(x=vix, y=spx, z=wealth, mode="markers",
                                 marker=dict(size=1.6, color=wealth, colorscale="Viridis",
                                             showscale=(col == 2)), showlegend=False),
                    row=1, col=col)
fig4b.update_layout(title="<b>QRH terminal wealth, simulated paths</b>", title_x=0.5,
                    height=650, scene=scene, scene2=scene)
fig4b.show()

In [ ]:
terminal_wealth_2_qrh  = portfolio_1_qrh.get_terminal_wealth()
terminal_wealth_20_qrh = portfolio_2_qrh.get_terminal_wealth()

kde_plot_qrh, ax = plt.subplots()
pal = sns.color_palette()
sns.kdeplot(terminal_wealth_2_qrh,  ax=ax, color=pal[0], label='lambda = 2')
sns.kdeplot(terminal_wealth_20_qrh, ax=ax, color=pal[1], label='lambda = 20')

# Histogram
ax.hist(terminal_wealth_2_qrh,  density=True, alpha=0.4)
ax.hist(terminal_wealth_20_qrh, density=True, alpha=0.4)
ax.set_xlabel("Terminal Wealth at Maturity")
ax.set_ylabel("Kernel Density Estimate")
ax.set_title("QRH")
ax.legend()
plt.show()

In [ ]:
def get_indifference_price_qrh( init_claim, strike, w_hat, side, lob_data=None):
    """
    Function to evaluate indifference price under the calibrated QRH model.
    lob_data selects the tradable set: vix_only_df reproduces the Part 2 universe,
    spx_vix_df adds the SPX quotes.
    """
    NO_SCENARIOS = 5000
    RISK_PARAM = 20
    spx_vix_df = lob_data if lob_data is not None else globals()["spx_vix_df"]
    if side == "BUY": # BUY CLAIM
        # Base portfolio
        P1 =  PortfolioOptimizerQRH(lob_data=spx_vix_df.copy(), qrh_calibration = qrh,
                                        wealth = 0.0, w_hat = w_hat,
                                        init_claim = init_claim, claim_sign = 1, claim_strike = None,
                                        risk_param = RISK_PARAM, no_scenarios = NO_SCENARIOS)
        # Portfolio that purchases claim
        P2 =  PortfolioOptimizerQRH(lob_data=spx_vix_df.copy(), qrh_calibration = qrh,
                                        wealth = 0.0, w_hat = w_hat,
                                        init_claim = init_claim, claim_sign = -1, claim_strike = strike,
                                        risk_param = RISK_PARAM, no_scenarios = NO_SCENARIOS)

    elif side == "SELL":  # SELL CLAIM
        # portfolio that sells claim
        P1 =  PortfolioOptimizerQRH(lob_data=spx_vix_df.copy(), qrh_calibration = qrh,
                                        wealth = 0.0, w_hat = w_hat,
                                        init_claim = init_claim, claim_sign = 1, claim_strike = strike,
                                        risk_param = RISK_PARAM, no_scenarios = NO_SCENARIOS)
        # Base portfolio
        P2 =  PortfolioOptimizerQRH(lob_data=spx_vix_df.copy(), qrh_calibration = qrh,
                                        wealth = 0.0, w_hat = w_hat,
                                        init_claim = init_claim, claim_sign = 1, claim_strike = None,
                                        risk_param = RISK_PARAM, no_scenarios = NO_SCENARIOS)
    else:
        raise ValueError("Side given isn't valid")

    P1.initialize_params()
    P2.initialize_params()

    P1.solve()
    P2.solve()

    indifference_px = w_hat*(P2.score() - P1.score())
    return indifference_px

In [ ]:
# Indifference prices under QRH, for two tradable universes:
#   vix_only_df -> same contracts as Part 2, so any difference is the MODEL
#   spx_vix_df  -> adds SPX, so the extra difference is the TRADABLES
qrh_prices = {}
for label, lob in [("QRH, VIX only", vix_only_df), ("QRH, VIX+SPX", spx_vix_df)]:
    buys, sells = [], []
    for K in strikes:
        buys.append(-1*get_indifference_price_qrh(init_claim=0, strike=K, w_hat=1e5,
                                                  side="BUY",  lob_data=lob))
        sells.append(-1*get_indifference_price_qrh(init_claim=0, strike=K, w_hat=1e5,
                                                   side="SELL", lob_data=lob))
    qrh_prices[label] = (buys, sells)
    print(f"{label}: done")

buy_prices_qrh, sell_prices_qrh = qrh_prices["QRH, VIX+SPX"]

In [ ]:
# Compare indifference prices: OU (Part 2) vs QRH (Part 3)
fig5 = go.Figure()
fig5.add_trace(go.Scatter(x=strikes, y=buy_prices,  mode='lines', name='BUY  - OU, VIX only',  line=dict(dash='dash')))
fig5.add_trace(go.Scatter(x=strikes, y=sell_prices, mode='lines', name='SELL - OU, VIX only',  line=dict(dash='dash')))
for label, (b, sl) in qrh_prices.items():
    fig5.add_trace(go.Scatter(x=strikes, y=b,  mode='lines', name=f'BUY  - {label}'))
    fig5.add_trace(go.Scatter(x=strikes, y=sl, mode='lines', name=f'SELL - {label}'))
fig5.update_layout(
    title="<b>Indifference price on Digital Option: OU vs QRH</b>",
    title_x = 0.5,
    xaxis_title="Digital Option Strike",
    yaxis_title="Indifference Price",
)
fig5.show()

# Buy-sell spread under both models
fig6 = go.Figure()
fig6.add_trace(go.Scatter(x=strikes, y=np.subtract(sell_prices, buy_prices), mode='lines',
                          name='OU, VIX only', line=dict(dash='dash')))
for label, (b, sl) in qrh_prices.items():
    fig6.add_trace(go.Scatter(x=strikes, y=np.subtract(sl, b), mode='lines', name=label))
fig6.update_layout(
    title="<b>Indifference Price Buy-Sell Spread: OU vs QRH</b>",
    title_x = 0.5,
    xaxis_title="<b> Option Strike Price</b>",
    yaxis_title="<b> Buy-Sell Spread </b>",
)
fig6.show()

In [ ]:
# Why the prices differ: the QRH right tail is much heavier than the OU lognormal
vix_qrh = portfolio_2_qrh.get_vix_prices()
vix_ou  = np.exp(np.random.default_rng(0).normal(mu_xi_T, std_xi_T, len(vix_qrh)))

tail = pd.DataFrame({
    "statistic": ["median", "p90", "p99", "p99.9", "P(VIX>35) %"],
    "OU":  [np.median(vix_ou),  np.percentile(vix_ou, 90),  np.percentile(vix_ou, 99),
            np.percentile(vix_ou, 99.9),  100*(vix_ou > 35).mean()],
    "QRH": [np.median(vix_qrh), np.percentile(vix_qrh, 90), np.percentile(vix_qrh, 99),
            np.percentile(vix_qrh, 99.9), 100*(vix_qrh > 35).mean()],
}).round(3)
print(tail.to_string(index=False))

fig7 = go.Figure()
fig7.add_trace(go.Histogram(x=vix_ou,  name="OU",  opacity=0.55, nbinsx=120, histnorm="probability density"))
fig7.add_trace(go.Histogram(x=vix_qrh, name="QRH", opacity=0.55, nbinsx=120, histnorm="probability density"))
fig7.update_layout(barmode="overlay", title="<b>VIX at maturity: OU vs QRH</b>", title_x=0.5,
                   xaxis_title="VIX at maturity", xaxis_range=[8, 60])
fig7.show()

In [ ]:
# One simulated (VIX, SPX) path under the calibrated QRH model
from pricing_lib.qrh_utils import KernelParams, impute_y0, simulate_qrh, vix_at_T

N_PATHS, STEPS_PER_DAY, SEED, PATH_ID = 5, 8, 7, 0

days    = (pd.Timestamp("2025-07-17") - qrh.quote_date).days
n_steps = days * STEPS_PER_DAY
delta   = (days / 365.0) / n_steps
params  = qrh.calibrated.as_array()
kernel, c = KernelParams(*params[:3]), params[3]

grid = np.arange(n_steps + 1) * delta
y0_curve, _ = impute_y0(grid, qrh.fwd_var_fn, kernel, c)

# record S at every step so we keep the whole path, not just the terminal value
spx_by_step, shocks = simulate_qrh(kernel, c, y0_curve, n_steps * delta, n_steps,
                                   N_PATHS, SEED, record_steps=range(1, n_steps + 1),
                                   xi_target=qrh.fwd_var_fn(grid))

# simulation gives S_t/S_0, so scale to a level with the quoted forward
spx_fwd  = spx_vix_df.loc[spx_vix_df.sym == "SPX", "forward"].iloc[0]
spx_path = spx_fwd * np.vstack([np.ones(N_PATHS)] +
                               [spx_by_step[n] for n in range(1, n_steps + 1)])

# VIX_t is a 30-day forward variance integral, so evaluate it on a coarser grid
vix_at   = np.arange(1, n_steps + 1)
vix_path = np.full((n_steps + 1, N_PATHS), np.nan)
u = np.linspace(0, 30/365, 500)
vix_path[0] = np.sqrt(np.trapezoid(qrh.fwd_var_fn(u), u) / (30/365)) * 100
for n in vix_at:
    vix_path[n], _ = vix_at_T(shocks[:n], kernel, c, n * delta, n, qrh.fwd_var_fn)

t_days = grid * 365
fig8 = make_subplots(specs=[[{"secondary_y": True}]])
fig8.add_trace(go.Scatter(x=t_days, y=vix_path[:, PATH_ID], name="VIX",
                          mode="lines+markers", marker=dict(size=3),
                          connectgaps=True, line=dict(color="crimson")), secondary_y=False)
fig8.add_trace(go.Scatter(x=t_days, y=spx_path[:, PATH_ID], name="SPX",
                          line=dict(color="royalblue")), secondary_y=True)
fig8.update_layout(title="<b>A simulated QRH path: VIX and SPX</b>", title_x=0.5,
                   xaxis_title="days from 21/06/2024", height=450)
fig8.update_yaxes(title_text="VIX", secondary_y=False)
fig8.update_yaxes(title_text="SPX", secondary_y=True)
fig8.show()